# درس 8: رگرسیون خطی

متن کامل درس: [https://quantacademy.ir/courses/quant-math/linear-regression/](https://quantacademy.ir/courses/quant-math/linear-regression/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. مدل و معادله‌های نرمال

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
market = np.log(close).diff().dropna().values
n = len(market)

rng = np.random.default_rng(31)
alpha_true, beta_true, noise_sd = 0.0002, 1.3, 0.02
y = alpha_true + beta_true * market + noise_sd * rng.standard_normal(n)

X = np.column_stack([np.ones(n), market])
coef = np.linalg.solve(X.T @ X, X.T @ y)
print(f"{coef[0]:.5f} {coef[1]:.5f}")

beta_cov = np.cov(market, y)[0, 1] / np.var(market, ddof=1)
alpha_cov = y.mean() - beta_cov * market.mean()
print(round(alpha_cov, 5), round(beta_cov, 5))
lsq = np.linalg.lstsq(X, y, rcond=None)[0]
print(f"{lsq[0]:.5f} {lsq[1]:.5f}")

## ۲. خطای استاندارد، t و R²

In [ ]:
resid = y - X @ coef
k = X.shape[1]
sigma2 = resid @ resid / (n - k)
cov_beta = sigma2 * np.linalg.inv(X.T @ X)
se = np.sqrt(np.diag(cov_beta))
t_stats = coef / se
p_vals = 2 * stats.t.sf(np.abs(t_stats), df=n - k)
r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))

for name, c, s, t, p in zip(["alpha", "beta"], coef, se, t_stats, p_vals):
    print(name, round(c, 5), round(s, 5), round(t, 2), round(p, 4))
print(round(r2, 3), round(np.sqrt(sigma2), 4))

lr = stats.linregress(market, y)
print(round(lr.slope, 5), round(lr.stderr, 5), round(lr.rvalue**2, 3))

## ۳. آلفا را نمی‌شود دید

## ۴. باقیمانده‌ها را وارسی کن

In [ ]:
print(f"{resid.mean():.1e}", round(resid.std(), 4))
print(round(stats.skew(resid), 2), round(stats.kurtosis(resid), 2))
dw = np.sum(np.diff(resid) ** 2) / np.sum(resid**2)
print(round(dw, 2))

rng = np.random.default_rng(32)
y_het = alpha_true + beta_true * market + 0.6 * np.abs(market) * rng.standard_normal(n)
coef_h = np.linalg.solve(X.T @ X, X.T @ y_het)
resid_h = y_het - X @ coef_h
print(round(np.corrcoef(np.abs(resid), np.abs(market))[0, 1], 3), round(np.corrcoef(np.abs(resid_h), np.abs(market))[0, 1], 3))

## ۵. بیش‌برازش: R² با افزودن متغیر بالا می‌رود

In [ ]:
rng = np.random.default_rng(33)
n_train, n_test = 60, 5000
rows = []
for k in [1, 5, 10, 20, 40]:
    Xtr = rng.standard_normal((n_train, k))
    Xte = rng.standard_normal((n_test, k))
    ytr = rng.standard_normal(n_train)
    yte = rng.standard_normal(n_test)
    A = np.column_stack([np.ones(n_train), Xtr])
    b = np.linalg.lstsq(A, ytr, rcond=None)[0]
    res = ytr - A @ b
    r2_in = 1 - (res @ res) / ((ytr - ytr.mean()) @ (ytr - ytr.mean()))
    adj = 1 - (1 - r2_in) * (n_train - 1) / (n_train - k - 1)
    pred = np.column_stack([np.ones(n_test), Xte]) @ b
    r2_out = 1 - ((yte - pred) @ (yte - pred)) / ((yte - yte.mean()) @ (yte - yte.mean()))
    rows.append((k, round(r2_in, 3), round(adj, 3), round(r2_out, 3)))
print(pd.DataFrame(rows, columns=["features", "R2_in", "R2_adj", "R2_out"]).to_string(index=False))

## ۶. هم‌خطی: ضرایب ناپایدار می‌شوند

In [ ]:
rng = np.random.default_rng(34)

def beta1_spread(corr, size=250, trials=3000):
    est1, est_sum = [], []
    for _ in range(trials):
        x1 = rng.standard_normal(size)
        x2 = corr * x1 + np.sqrt(1 - corr**2) * rng.standard_normal(size)
        target = x1 + x2 + rng.standard_normal(size)
        A = np.column_stack([np.ones(size), x1, x2])
        b = np.linalg.lstsq(A, target, rcond=None)[0]
        est1.append(b[1])
        est_sum.append(b[1] + b[2])
    return float(round(np.mean(est1), 2)), float(round(np.std(est1), 3)), float(round(np.std(est_sum), 3))

for corr in [0.0, 0.9, 0.99]:
    mean1, sd1, sd_sum = beta1_spread(corr)
    print(corr, mean1, sd1, sd_sum)